# День 7 — Анализ ошибок и мини-приложение

Разбираем ошибки fine-tuned модели из Дня 5, сравниваем метрики с baseline
из Дня 4 и запускаем демо через FastAPI (`api.py`).

## Задачи 1–2: False Positive / False Negative

Предсказания пересчитываются на той же выборке, что и в Дне 6:
`data/imdb_test.csv`, 500 строк, `random_state=42`.

In [ ]:
from pathlib import Path

import pandas as pd
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

from sentiment_lab.baseline import load_model as load_baseline_model
from sentiment_lab.inference import predict_baseline, predict_fine_tuned

model_dir = Path("models/fine_tuned_model")
model_ft = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model_ft.eval()

baseline_model = load_baseline_model()
encoder_model = AutoModel.from_pretrained("distilbert-base-uncased")
encoder_model.eval()

test_df = pd.read_csv("data/imdb_test.csv").sample(n=500, random_state=42)
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

y_pred_ft = [p.prediction for p in predict_fine_tuned(test_texts, model_ft, tokenizer)]
y_pred_base = [
    p.prediction
    for p in predict_baseline(test_texts, baseline_model, tokenizer, encoder_model)
]
print(f'Предсказаний: {len(y_pred_ft)} / {len(y_pred_base)}')

In [ ]:
from sentiment_lab.error_analysis import analyze_errors, save_error_report

analysis = analyze_errors(test_texts, test_labels, y_pred_ft)
fp = analysis.false_positives
fn = analysis.false_negatives
print(f'Всего ошибок: {len(analysis.errors)}')
print(f'False Positives: {len(fp)}')
print(f'False Negatives: {len(fn)}')

print('\n=== FALSE POSITIVES (сказали good, а было bad) ===')
for _, row in fp.head(5).iterrows():
    print(f'\nТекст: {row["text"][:100]}...')
    print(f'Истинный класс: {row["true_label"]}, Предсказан: {row["pred_label"]}')

print('\n=== FALSE NEGATIVES (сказали bad, а было good) ===')
for _, row in fn.head(5).iterrows():
    print(f'\nТекст: {row["text"][:100]}...')
    print(f'Истинный класс: {row["true_label"]}, Предсказан: {row["pred_label"]}')

print(f'\nСредняя длина ошибочных текстов: {analysis.mean_error_length:.0f}')
print(f'Средняя длина всех текстов: {analysis.mean_all_length:.0f}')
print(f'Отчёт: {save_error_report(analysis)}')

## Задачи 6–7: метрики и comparison_results.txt

In [ ]:
from sklearn.metrics import classification_report

from sentiment_lab.evaluation import compute_metrics, save_comparison_results

print('=== Fine-tuned Model ===')
print(classification_report(test_labels, y_pred_ft))
metrics_ft = compute_metrics(test_labels, y_pred_ft)

print('=== Baseline Model ===')
print(classification_report(test_labels, y_pred_base))
metrics_base = compute_metrics(test_labels, y_pred_base)

print('=== Сравнение ===')
print(f'Fine-tuned F1: {metrics_ft.macro_f1:.4f}, Accuracy: {metrics_ft.accuracy:.4f}')
print(f'Baseline F1: {metrics_base.macro_f1:.4f}, Accuracy: {metrics_base.accuracy:.4f}')
improvement = (metrics_ft.macro_f1 - metrics_base.macro_f1) / metrics_base.macro_f1 * 100
print(f'Улучшение F1: {improvement:.2f}%')
print(f'Файл: {save_comparison_results(metrics_ft, metrics_base)}')

## Демо через FastAPI

```bash
uv sync --group demo
uv run uvicorn api:app --reload
```

POST http://127.0.0.1:8000/predict с телом `{"text": "Great movie!"}`.